# Day 25: LLM Orchestration and FDE Skills 3b
**Tool calling, LangGraph workflows and systems thinking**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas langgraph matplotlib

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas langgraph matplotlib
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Tools: schemas, validation and an allowlist

In [ ]:
import json, re
from pydantic import BaseModel, Field, ValidationError
import genai_helpers

ORDERS = {"ORD-1042": {"status": "delivered", "amount": 4800, "days_late": 3},
          "ORD-2077": {"status": "in transit", "amount": 1200, "days_late": 0}}

class OrderArgs(BaseModel):
    order_id: str = Field(pattern=r"^ORD-\d{4}$")

class RefundArgs(BaseModel):
    amount: float = Field(gt=0)
    days_late: int = Field(ge=0)

def get_order_status(order_id):
    o = ORDERS.get(order_id)
    return json.dumps(o if o else {"error": "order not found"})

def calculate_refund(amount, days_late):
    pct = min(0.05 * days_late, 0.5)             # 5% per day late, capped at 50%
    return json.dumps({"refund": round(amount * pct, 2)})

REGISTRY = {"get_order_status": (get_order_status, OrderArgs), "calculate_refund": (calculate_refund, RefundArgs)}

TOOLS = [
    {"type": "function", "function": {"name": "get_order_status",
        "description": "Look up an order by id, for example ORD-1042. Returns status, amount and days late.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "calculate_refund",
        "description": "Calculate a late delivery refund from the order amount and days late.",
        "parameters": {"type": "object", "properties": {"amount": {"type": "number"}, "days_late": {"type": "integer"}},
                       "required": ["amount", "days_late"]}}},
]

def run_tool(name, args_json):
    if name not in REGISTRY:
        return json.dumps({"error": f"unknown tool {name}"})
    fn, schema = REGISTRY[name]
    try:
        args = schema(**json.loads(args_json))
    except (ValidationError, json.JSONDecodeError) as e:
        return json.dumps({"error": f"invalid arguments: {e.errors()[0]['msg'] if hasattr(e, 'errors') else e}"})
    return fn(**args.model_dump())

print(run_tool("delete_database", "{}"))
print(run_tool("get_order_status", '{"order_id": "1042"}'))
print(run_tool("calculate_refund", '{"amount": 4800, "days_late": 3}'))

## 3. The tool calling loop

In [ ]:
def offline_tool_model(messages):
    # Stand-in for a model that supports tool calling: decides the next step from what it has seen so far.
    user = messages[0]["content"]
    results = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    oid = re.search(r"ORD-\d{4}", user)
    if not results and oid:
        return {"tool": "get_order_status", "args": json.dumps({"order_id": oid.group(0)})}
    if len(results) == 1 and "amount" in results[0]:
        return {"tool": "calculate_refund", "args": json.dumps({"amount": results[0]["amount"], "days_late": results[0]["days_late"]})}
    if results and "refund" in results[-1]:
        return {"answer": f"Your refund is {results[-1]['refund']} INR for the late delivery."}
    return {"answer": "I could not find that order. Please check the order id."}

def ask_with_tools(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        if genai_helpers.ONLINE:
            r = genai_helpers.client.chat.completions.create(model=genai_helpers.MODEL, messages=messages, tools=TOOLS)
            m = r.choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append(m)
            for call in m.tool_calls:
                out = run_tool(call.function.name, call.function.arguments)
                print(f"step {step + 1}: {call.function.name}({call.function.arguments}) -> {out}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": out})
        else:
            d = offline_tool_model(messages)
            if "answer" in d:
                return d["answer"]
            out = run_tool(d["tool"], d["args"])
            print(f"step {step + 1}: {d['tool']}({d['args']}) -> {out}")
            messages.append({"role": "tool", "content": out})
    return "Stopped: too many steps"

print("ANSWER:", ask_with_tools("ORD-1042 arrived late. What refund do I get?"))
print()
print("ANSWER:", ask_with_tools("Where is order ORD-9999?"))

### TODO 1: Add a risky `issue_refund(order_id, amount)` tool that asks for confirmation before running. Use an `AUTO_APPROVE` flag so tests can run without typing.

In [ ]:
# TODO: write your code here

## 4. A LangGraph workflow with branches and a retry loop

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

POLICY = {"refund": "Late deliveries get 5% of the order value per day late, up to 50%.",
          "return": "Items can be returned within 30 days in original packaging."}

class State(TypedDict, total=False):
    question: str
    intent: str
    evidence: str
    answer: str
    attempts: int
    path: list

def classify(s):
    q = s["question"].lower()
    intent = "order" if re.search(r"ord-\d{4}", q) else "policy" if any(w in q for w in ["refund", "return", "policy", "late"]) else "other"
    return {"intent": intent, "attempts": s.get("attempts", 0), "path": s.get("path", []) + [f"classify:{intent}"]}

def lookup_order(s):
    oid = re.search(r"ORD-\d{4}", s["question"], re.I).group(0).upper()
    return {"evidence": get_order_status(oid), "path": s["path"] + ["lookup_order"]}

def retrieve_policy(s):
    q = s["question"].lower()
    hits = [v for k, v in POLICY.items() if k in q]
    return {"evidence": " ".join(hits), "path": s["path"] + ["retrieve_policy"]}

def answer(s):
    if not s.get("evidence"):
        return {"answer": "", "path": s["path"] + ["answer:empty"]}
    a = llm(f"Context: {s['evidence']}\nQuestion: {s['question']}", system="Answer only from the context in one sentence.")
    return {"answer": a, "path": s["path"] + ["answer"]}

def check(s):
    return {"attempts": s["attempts"] + 1, "path": s["path"] + ["check"]}

def rewrite(s):
    return {"question": s["question"] + " (refund policy)", "path": s["path"] + ["rewrite"]}

def human(s):
    return {"answer": "Handed to a human agent.", "path": s["path"] + ["human"]}

def route_intent(s):
    return {"order": "lookup_order", "policy": "retrieve_policy"}.get(s["intent"], "human")

def route_check(s):
    if s.get("answer") and "don't know" not in s["answer"].lower():
        return "done"
    return "retry" if s["attempts"] < 2 else "human"

g = StateGraph(State)
for name, fn in [("classify", classify), ("lookup_order", lookup_order), ("retrieve_policy", retrieve_policy),
                 ("answer", answer), ("check", check), ("rewrite", rewrite), ("human", human)]:
    g.add_node(name, fn)
g.set_entry_point("classify")
g.add_conditional_edges("classify", route_intent, {"lookup_order": "lookup_order", "retrieve_policy": "retrieve_policy", "human": "human"})
g.add_edge("lookup_order", "answer")
g.add_edge("retrieve_policy", "answer")
g.add_edge("answer", "check")
g.add_conditional_edges("check", route_check, {"done": END, "retry": "rewrite", "human": "human"})
g.add_edge("rewrite", "retrieve_policy")
g.add_edge("human", END)
app = g.compile()

for q in ["What is the status of ORD-2077?", "How much do I get if delivery is late?", "Can you sing a song?"]:
    out = app.invoke({"question": q})
    print(f"Q: {q}\n   path: {' -> '.join(out['path'])}\n   answer: {out['answer']}\n")

In [ ]:
print(app.get_graph().draw_mermaid())

## 5. Systems thinking: simulate the chatbot launch
A customer launched a chatbot. Some bot answers are wrong, which creates follow-up tickets. Updating the knowledge base takes time (a delay). Try different leverage points.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def simulate(weeks=12, review_failed_answers=False, extra_agents=0):
    backlog, wrong_rate, history = 200.0, 0.30, []
    pending_fixes = [0.0, 0.0]                 # knowledge base fixes arrive 2 weeks later (delay)
    for w in range(weeks):
        new = 400.0
        bot_handled = new * 0.5
        follow_ups = bot_handled * wrong_rate           # reinforcing: wrong answers create more tickets
        capacity = 250 + 40 * extra_agents
        backlog = max(0.0, backlog + (new - bot_handled) + follow_ups - capacity + 0.1 * backlog)  # long queues cause chasers
        fix = 0.04 if review_failed_answers else 0.0
        pending_fixes.append(fix)
        wrong_rate = max(0.05, wrong_rate - pending_fixes.pop(0))
        history.append(backlog)
    return history

scenarios = {"do nothing": simulate(), "hire 2 agents": simulate(extra_agents=2),
             "review failed bot answers": simulate(review_failed_answers=True)}
for name, h in scenarios.items():
    plt.plot(range(1, 13), h, marker="o", label=name)
plt.title("Support backlog over 12 weeks after the chatbot launch")
plt.xlabel("Week"); plt.ylabel("Open tickets"); plt.legend(); plt.grid(alpha=0.3); plt.show()
for name, h in scenarios.items():
    print(f"{name:<26} week 12 backlog: {h[-1]:.0f}")

### TODO 2: Add your own leverage point to `simulate` (for example, routing billing questions straight to humans) and compare it with the others.

In [ ]:
# TODO: write your code here

---
**Yoi shuumatsu wo** (良い週末を): *Have a great weekend*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)